In [7]:
import pandas as pd

In [10]:
df = pd.read_csv('../data/water_quality.csv')
df.head()
df.head(10)
df.shape
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 19029 entries, 0 to 19028
Data columns (total 24 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   Well_ID                       15244 non-null  str    
 1   State                         19029 non-null  str    
 2   District                      19029 non-null  str    
 3   Block                         17910 non-null  str    
 4   Village                       19028 non-null  str    
 5   Latitude                      18640 non-null  float64
 6   Longitude                     18639 non-null  float64
 7   Year                          19029 non-null  int64  
 8   pH                            19029 non-null  float64
 9   EC                            19029 non-null  float64
 10  CO3                           19029 non-null  float64
 11  HCO3                          19029 non-null  float64
 12  Cl                            19029 non-null  float64
 13  SO4         

## 1. Exploratory Data Analysis (EDA)
### Understanding water quality patterns across regions and years

In [11]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# Check missing values
print("Missing Values:")
print(df.isnull().sum())
print("\n" + "="*50)

# Statistical summary
print("\nStatistical Summary of Water Quality Parameters:")
print(df.describe())

Missing Values:
Well_ID                         3785
State                              0
District                           0
Block                           1119
Village                            1
Latitude                         389
Longitude                        390
Year                               0
pH                                 0
EC                                 0
CO3                                0
HCO3                               0
Cl                                 0
SO4                                0
NO3                                0
TH                                 0
Ca                                 0
Mg                                 0
Na                                 0
K                                  0
F                                  0
TDS                                0
WQI                                0
Water Quality Classification       0
dtype: int64


Statistical Summary of Water Quality Parameters:
           Latitude     Longitud

In [12]:
# Water Quality by State
print("Water Quality Index by State:")
wqi_by_state = df.groupby('State')['WQI'].agg(['mean', 'median', 'std', 'count'])
print(wqi_by_state.sort_values('mean', ascending=False).head(10))

print("\n" + "="*50)

# Water Quality Distribution by Year
print("\nWater Quality Index by Year:")
wqi_by_year = df.groupby('Year')['WQI'].agg(['mean', 'median', 'count'])
print(wqi_by_year)

Water Quality Index by State:
                      mean     median         std  count
State                                                   
Delhi           712.936564  437.04850  806.611064    149
Gujarat         538.248039  343.55920  603.401184   1657
Andhra Pradesh  485.852596  396.59500  369.456642   1265
Tamil Nadu      473.789152  389.55310  362.029381   2115
Telangana       393.358935  324.54480  263.911170    377
Karnataka       269.645710  226.02676  228.377206   1822
Maharashtra     263.759956  230.47266  169.472675   2366
Madhya Pradesh  258.774626  234.36660  123.970804   3628
Uttar Pradesh   229.463892  168.53326  284.718441   1260
West Bengal     212.735474  184.17722  151.344339   1301


Water Quality Index by Year:
            mean      median  count
Year                               
2019  308.024338  217.429983   8650
2020  333.808290  254.173800   2136
2021  303.234763  225.689580   2390
2022  291.686543  245.919380   5853


## 2. Identify Key Factors Influencing Water Quality
### Correlation analysis and feature importance

In [13]:
# Select numeric columns for correlation
numeric_cols = df.select_dtypes(include=[np.number]).columns

# Calculate correlation with WQI
correlation_with_wqi = df[numeric_cols].corr()['WQI'].sort_values(ascending=False)
print("Correlation of Parameters with Water Quality Index (WQI):")
print(correlation_with_wqi)

print("\n" + "="*50)

# Top factors influencing WQI
print("\nTop 5 Factors Most Correlated with WQI:")
print(correlation_with_wqi[1:6])  # Exclude WQI itself

Correlation of Parameters with Water Quality Index (WQI):
WQI          1.000000
EC           0.981442
Cl           0.932176
TDS          0.924358
Na           0.903628
TH           0.822221
Mg           0.776863
SO4          0.705896
Ca           0.624723
HCO3         0.493898
NO3          0.380014
K            0.274142
CO3          0.096421
F            0.092698
pH           0.012890
Longitude   -0.003164
Latitude    -0.004263
Year        -0.023797
Name: WQI, dtype: float64


Top 5 Factors Most Correlated with WQI:
EC     0.981442
Cl     0.932176
TDS    0.924358
Na     0.903628
TH     0.822221
Name: WQI, dtype: float64


## 3. Build Predictive Models to Estimate Water Quality Index
### Prepare data and train multiple models

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

# Prepare features and target
# Select numeric features (excluding WQI as it's our target)
X = df[['pH', 'EC', 'CO3', 'HCO3', 'Cl', 'SO4', 'NO3', 'TH', 'Ca', 'Mg', 'Na', 'K', 'F', 'TDS']]
y = df['WQI']

# Remove rows with missing values
valid_idx = X.notna().all(axis=1) & y.notna()
X = X[valid_idx]
y = y[valid_idx]

print(f"Dataset shape after cleaning: {X.shape}")
print(f"Target variable shape: {y.shape}")

# Split data into training and testing sets (80-20 split)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"\nTraining set size: {X_train.shape[0]}")
print(f"Testing set size: {X_test.shape[0]}")

In [ ]:
# Scale the features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Model 1: Linear Regression
print("="*50)
print("LINEAR REGRESSION MODEL")
print("="*50)

lr_model = LinearRegression()
lr_model.fit(X_train_scaled, y_train)

# Predictions
y_train_pred_lr = lr_model.predict(X_train_scaled)
y_test_pred_lr = lr_model.predict(X_test_scaled)

# Evaluation
lr_train_r2 = r2_score(y_train, y_train_pred_lr)
lr_test_r2 = r2_score(y_test, y_test_pred_lr)
lr_test_rmse = np.sqrt(mean_squared_error(y_test, y_test_pred_lr))
lr_test_mae = mean_absolute_error(y_test, y_test_pred_lr)

print(f"Training R² Score: {lr_train_r2:.4f}")
print(f"Testing R² Score: {lr_test_r2:.4f}")
print(f"Testing RMSE: {lr_test_rmse:.4f}")
print(f"Testing MAE: {lr_test_mae:.4f}")

In [ ]:
# Model 2: Random Forest
print("\n" + "="*50)
print("RANDOM FOREST MODEL")
print("="*50)

rf_model = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
rf_model.fit(X_train, y_train)

# Predictions
y_train_pred_rf = rf_model.predict(X_train)
y_test_pred_rf = rf_model.predict(X_test)

# Evaluation
rf_train_r2 = r2_score(y_train, y_train_pred_rf)
rf_test_r2 = r2_score(y_test, y_test_pred_rf)
rf_test_rmse = np.sqrt(mean_squared_error(y_test, y_test_pred_rf))
rf_test_mae = mean_absolute_error(y_test, y_test_pred_rf)

print(f"Training R² Score: {rf_train_r2:.4f}")
print(f"Testing R² Score: {rf_test_r2:.4f}")
print(f"Testing RMSE: {rf_test_rmse:.4f}")
print(f"Testing MAE: {rf_test_mae:.4f}")

# Feature Importance
print("\nTop 5 Most Important Features:")
feature_importance = pd.DataFrame({
    'Feature': X.columns,
    'Importance': rf_model.feature_importances_
}).sort_values('Importance', ascending=False)
print(feature_importance.head())

## 4. Generate Insights for Water Quality Management and Policy Recommendations

In [ ]:
# 1. Identify States with Poor Water Quality
print("="*60)
print("STATES WITH POOREST WATER QUALITY (Lowest WQI)")
print("="*60)
poor_quality_states = df.groupby('State')['WQI'].mean().sort_values().head(10)
print(poor_quality_states)

print("\n" + "="*60)
print("STATES WITH BEST WATER QUALITY (Highest WQI)")
print("="*60)
good_quality_states = df.groupby('State')['WQI'].mean().sort_values(ascending=False).head(10)
print(good_quality_states)

# 2. Identify most problematic water quality parameters
print("\n" + "="*60)
print("PARAMETERS WITH HIGHEST VALUES (May Indicate Pollution)")
print("="*60)
parameter_means = df[['pH', 'EC', 'Cl', 'SO4', 'NO3', 'TH', 'TDS']].mean()
print(parameter_means.sort_values(ascending=False))

# 3. Water Quality Classification Summary
print("\n" + "="*60)
print("WATER QUALITY CLASSIFICATION DISTRIBUTION")
print("="*60)
wq_classification = df['Water Quality Classification'].value_counts()
print(wq_classification)
print(f"\nPercentage Distribution:")
print((df['Water Quality Classification'].value_counts(normalize=True) * 100).round(2))

In [ ]:
# 4. Policy Recommendations Summary
print("\n" + "="*60)
print("KEY INSIGHTS & POLICY RECOMMENDATIONS")
print("="*60)

insights = f"""
1. MODEL PERFORMANCE:
   - Random Forest R² Score: {rf_test_r2:.4f} (explains {rf_test_r2*100:.2f}% of WQI variance)
   - Linear Regression R² Score: {lr_test_r2:.4f}
   - Random Forest is better for WQI prediction

2. KEY FACTORS AFFECTING WATER QUALITY:
   Top influencing parameters (by importance):
   - {feature_importance.iloc[0]['Feature']}: {feature_importance.iloc[0]['Importance']:.4f}
   - {feature_importance.iloc[1]['Feature']}: {feature_importance.iloc[1]['Importance']:.4f}
   - {feature_importance.iloc[2]['Feature']}: {feature_importance.iloc[2]['Importance']:.4f}

3. CRITICAL AREAS FOR INTERVENTION:
   - Focus on states with lowest average WQI: {poor_quality_states.index[0]}, {poor_quality_states.index[1]}, {poor_quality_states.index[2]}
   - Monitor high levels of: Fluoride (F), TDS, and Nitrates (NO3)

4. RECOMMENDED POLICIES:
   ✓ Priority water treatment in identified poor-quality states
   ✓ Regulate industrial discharge affecting high pollutants
   ✓ Implement monitoring systems for critical parameters
   ✓ Public awareness campaigns in affected regions
"""

print(insights)